# Goal 5 · Graph query guard

**Goal:** harden the graph read tool with an application-level guard that admits genuine read queries and rejects administrative, procedure and write clauses.

**File:** `src/mcp_server.py` (`execute_read_cypher`)  |  **Evidence:** `cypher_guard_result.json`

### Concepts

- **The tool.** `execute_read_cypher` hands **model-written Cypher** to Neo4j.
- **What read mode does and doesn't block.** The session runs with `default_access_mode="r"` (`execute_read`), and the database rejects writes in a read transaction. Read mode does **not** stop:
  - procedure calls (`CALL dbms.*`, `CALL db.*`, APOC), which expose server internals, configuration, users and indexes;
  - expensive scans, and anything a future procedure plugin exposes.
- **Defence in depth.** Validate the query **before** it reaches the database:
  1. **Allow-list the shape.** A read must start with `MATCH`, `OPTIONAL MATCH`, `WITH`, `UNWIND` or `RETURN`, and must `RETURN` something.
  2. **Deny dangerous clauses.** Reject any query containing a token from `_FORBIDDEN_CYPHER` (already defined at the top of the file).
  3. Answer refusals with a string starting with **`Rejected:`**. Callers and the evidence probe look for that prefix.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))
import labkit
mcp_server = labkit.reload('mcp_server')
print('denylist:', mcp_server._FORBIDDEN_CYPHER)

# The write probes target a label that matches no nodes, so they're harmless even before the guard exists
PROBES = [   # (query, should_be_admitted)
    ('MATCH (c:OntologyClass) RETURN c.name', True),                                   # the evidence's allowed probe
    ('match (m:GovernedMetric)-[:COMPUTED_OVER]->(c) return m.name, c.name', True),    # lower case is still a read
    ('OPTIONAL MATCH (e:Entity) RETURN count(e) AS entities', True),
    ('UNWIND [1, 2, 3] AS x RETURN x', True),
    ('CALL dbms.components() YIELD name RETURN name', False),                          # the evidence's blocked probe
    ('call db.labels()', False),
    ('MATCH (n) RETURN apoc.version()', False),
    ('MATCH (n:GuardProbeNoSuchLabel) DETACH DELETE n', False),
    ('MATCH (e:GuardProbeNoSuchLabel) SET e.name = "pwned" RETURN e', False),
    ('LOAD CSV FROM "file:///claims_export.csv" AS row RETURN row', False),
    ('MATCH (c:OntologyClass) WITH c CALL { MATCH (x) RETURN count(x) AS n } RETURN c.name, n', False),
    ('SHOW USERS', False),
]

def run_probes():
    ok_all = True
    for q, should_admit in PROBES:
        resp = mcp_server.execute_read_cypher(q)
        admitted = not resp.startswith('Rejected:')
        ok = admitted == should_admit
        ok_all &= ok
        print(f"{'PASS' if ok else 'FAIL'}  {'admit ' if should_admit else 'reject'}  {q[:70]:70}  -> {resp.splitlines()[0][:70]}")
    print('\nall probes as expected' if ok_all else '\nsome probes are wrong')

run_probes()

### Observe

Before the fix, `CALL dbms.components()` **runs** and returns the server's name, version and edition. That's reconnaissance handed to whoever controls the prompt.

Writes already fail, but only at the database, as a `Security/Syntax Error`. They aren't refused by the tool's own policy.

## Task 5 · Guard `execute_read_cypher`

Replace the TODO comment block at the top of the function so that, **before** the query runs:

1. The query is normalised: whitespace collapsed to single spaces, upper-cased.
2. Anything not starting with `MATCH`, `OPTIONAL MATCH`, `WITH`, `UNWIND` or `RETURN` is rejected.
3. Anything without `RETURN` is rejected.
4. Anything containing a `_FORBIDDEN_CYPHER` token is rejected, naming the clause.

Each rejection returns a string starting with `Rejected:`.

In [ ]:
labkit.show('src/mcp_server.py', 'def execute_read_cypher', after=22)

<details><summary>Hint 1</summary><code>" ".join(cypher_query.split()).upper()</code> collapses tabs and newlines, so <code>CALL\n db.labels()</code> can't slip past a <code>"CALL "</code> check.</details>
<details><summary>Hint 2</summary><code>str.startswith</code> accepts a tuple of prefixes.</details>
<details><summary>Hint 3</summary>Loop over <code>_FORBIDDEN_CYPHER</code> and test <code>token in normalized</code>. Report <code>token.strip()</code> in the message.</details>

Save, then re-run the probe table. Every line should read PASS.

In [ ]:
mcp_server = labkit.reload('mcp_server')
run_probes()

## Publish the evidence

The evidence uses two fixed probes: one allowed read, and `CALL dbms.components()`. It records whether each was handled correctly.

In [ ]:
!{sys.executable} -m src.emit_evidence cypher-guard
labkit.upload('cypher_guard_result.json')
labkit.grade(5)

### Debrief: the limits of a substring denylist

The guard matches **substrings**, so it has false positives. Try it:

In [ ]:
for q in ['MATCH (c:Chunk) RETURN c.id, c.created_at',            # contains CREATE
          'MATCH (e:Entity) WHERE e.name = "Call Centre" RETURN e',   # contains CALL
          'MATCH (m:GovernedMetric) RETURN m.properties_used']:       # fine
    print(f'{q:62} -> {mcp_server.execute_read_cypher(q).splitlines()[0][:60]}')

Keep the lab's guard as it is: the grader's probes expect it. In production you would harden it further:
- match **whole words** outside string literals (a tokenizer, or `\b` regexes on a copy with literals removed);
- or ask the database: `EXPLAIN <query>` returns the plan without executing it, and you can reject plans that contain procedure calls or writes;
- add **database-side least privilege**: a dedicated read-only Neo4j user (Enterprise RBAC can deny `EXECUTE PROCEDURE` and restrict labels);
- bound the cost: the tool already truncates to 50 rows, and a `LIMIT` and a transaction timeout would bound scans too.

## All goals

Run the grader over everything you've published:

In [ ]:
labkit.grade()